**Sequential Chains**

We will create a system for helping people learn new skills. This system needs to be built sequentially, so learners can modify plans based on their preferences and constraints. You'll utilize your LangChain LCEL skills to build a sequential chain to build this system, and the first step is to design the prompt templates that will be used by this system.

In [2]:
# For the official Google SDK
#!pip install -q -U google-generativeai

# For LangChain integration
#!pip install -q -U langchain-google-genai

#For load_tools
#!pip install langchain-community

#For wikipedia library from python
#!pip install wikipedia

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=0956315bb78c2d70973e84fe5611671f5df21280ef1adf4221b4a5bb8381a7b2
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia


In [10]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, FewShotPromptTemplate
import google.genai as genai
from langchain_core.output_parsers import StrOutputParser
from langchain.agents import create_agent
from langchain_community.agent_toolkits.load_tools import load_tools
import pandas as pd

In [4]:
from google.colab import userdata
API_KEY = userdata.get('API_KEY')

In [5]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", api_key=API_KEY)

#Create a prompt template that takes an input activity
learning_prompt = PromptTemplate(
    input_variables=["activity"],
    template="I want to learn {activity}. Can you please suggest how i can learn this step by step? "
)

#Create a prompt template that places a time constraint on the output
time_prompt = PromptTemplate(
    input_variables=['learning_plan'],
    template = "I have only one week. Can you please create a plan to help me hit this goal? : {learning_plan}"
)

#Invoke the learning plan with an activity
print(learning_prompt.invoke({"activity": "play golf"}))

text='I want to learn play golf. Can you please suggest how i can learn this step by step? '


In [10]:
#Complete the sequence chain with LCEL
seq_chain = ({"learning_plan":learning_prompt| llm | StrOutputParser()}) | time_prompt |llm |StrOutputParser()

#Call the chain
print(seq_chain.invoke({"activity":"learn spanish"}))

¡Excelente elección! Given you only have **one week**, we need to be incredibly focused and realistic. The goal for this week won't be fluency, but rather to achieve **"Survival Spanish" (A1 Level)** – enough to introduce yourself, ask/answer very basic questions, understand common phrases, and get a foundational feel for the language.

This plan is an intensive sprint, requiring significant dedication each day. Aim for **2-4 hours of focused study daily**, broken into manageable chunks.

---

## Your 1-Week Intensive Spanish Sprint: Survival A1

**Overall Goal:** By the end of the week, you should be able to:
*   Pronounce Spanish words with reasonable accuracy.
*   Introduce yourself and ask someone's name/origin.
*   Understand and use basic greetings, goodbyes, and courtesy phrases.
*   Count from 1-100.
*   Form very simple sentences using "Ser" and "Estar" and a few regular verbs.
*   Ask and understand simple "who, what, where, how" questions.

---

### Key Resources for the Wee

Running a series of chains opens up lots of possibility for designing more sophisticated workflows. Now we're beginning to grasp how LangChain can handle more sophisticated workflows, it's time to talk about agents, which enable LLMs to make decisions

**LangChain Agents**

**ReAct Agent**

 ReAct stands for Reason and Act, which describes how they make decisions. Here, we'll load the built-in wikipedia tool to integrate external data from Wikipedia with your LLM.

In [9]:
#Define tools
tools = load_tools(["wikipedia"])

#Define the agent
agent = create_agent(llm,tools)

#Invoke the agent
response = agent.invoke({"messages":[("human","How many people live in sydney?")]})
print(response['messages'][-1].content)

[{'type': 'text', 'text': 'The estimated population of Sydney in June 2024 was 5,557,233.', 'extras': {'signature': 'CtsBAb4+9vumgMsmA4pgrry/T1+pBmPKhSq917+bLoU/EohBBT7GtTRgP81T5elD1eJkKX1CdXBot6bWFymsIz2a3a8cOc82DQqsBEIFe0QPETqGilBkdMQC8NkfI+4bix7ydiI4jIZGoLW7btptInGZ+YkZSWUTgPcXme9+n7t1r2e5w4k096GuHnnKSaS8+yoV3xWwOJwDZ5QwIraVx3cRtUKR08+JDWbSSMAEy+HBGa+Wo8Y92F8D95zBKKc2ciAv9bI9Y9euzBPw1hl9pQRvGhl42Z9iOZdEV+QT'}}]


 The wikipedia tool is pretty handy for quickly looking up specific facts to enter into your workflows, potentially overcoming an LLM's knowledge gap

**Custom tool for Agents**

Lets say we're working for a SaaS (software as a service) company with big goals for rolling out tools to help employees at all levels of the organization to make data-informed decisions. We're creating a PoC for an application that allows customer success managers to interface with company data using natural language to retrieve important customer data.

In [11]:
id = [101,102,103,104,105,106,107,108,109,110]
name = ['Tech Innovators Inc.', 'Green Solutions Ltd.', 'Global Enterprises', 'Peak Performance Co.', 'Visionary Ventures','NextGen Technologies', 'Dynamic Dynamics LLC','Infinity Services','Eco-Friendly Products', 'Future Insights']
subscription_type = ['Premium', 'Standard', 'Basic','Premium', 'Standard', 'Basic','Premium', 'Standard', 'Basic','Premium']
active_users = [450,300,150,800,600,200,700,500,100,900]
auto_renewals = [True,True,False,True,True,False,True,True,False,True]

data = list(zip(id, name, subscription_type, active_users, auto_renewals))
customers = pd.DataFrame(data, columns=['id', 'name', 'subscription_type', 'active_users', 'auto_renewals'])

In [12]:
#Define a fucntion to retrive customer info by name
def retrieve_customer_info(name:str)->str:
  """Retrieve customer information by name """
  #Filter customers for the customers name
  customer_info = customers[customers["name"]==name]
  return customer_info.to_string()

#Call the function on Peak Performance Co.
print(retrieve_customer_info("Peak Performance Co."))


    id                  name subscription_type  active_users  auto_renewals
3  104  Peak Performance Co.           Premium           800           True


In [14]:
from langchain.tools import tool

#Convert the retrive_customer_info into a tool
@tool
def retrieve_customer_info(name:str)->str:
  """Retrieve customer information by name """
  #Filter customers for the customers name
  customer_info = customers[customers["name"]==name]
  return customer_info.to_string()

#Print the customer tool arguments
print(retrieve_customer_info.args)

{'name': {'title': 'Name', 'type': 'string'}}


The @tool decorator is a really nice way to quickly convert Python functions into custom tools that are compatible with LangChain agents.

In [15]:
#Creat a ReAct agent
agent = create_agent(llm, [retrieve_customer_info])

#Invoke the agent on the input
messages = agent.invoke({"messages":[("human","Create a summary of our customer: Peak Performance Co.")]})
print(messages["messages"][-1].content)

Here's a summary of Peak Performance Co.:

*   **ID:** 104
*   **Name:** Peak Performance Co.
*   **Subscription Type:** Premium
*   **Active Users:** 800
*   **Auto-Renewals:** True


Imagine the value an application like this could bring to an organization.Natural language interfaces to databases and other data sources are becoming more common, lowering the barrier to entry for making data-informed decisions, all thanks to LLMs and tools like LangChain

In [16]:
!pip freeze > requirements.txt